# `mdpp` Example: Structure Metrics

This notebook covers trajectory-level structural metrics:

- `compute_rmsd`
- `compute_rmsf`
- `compute_sasa`
- `compute_radius_of_gyration`

Use this notebook after the IO/alignment workflow.
            


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from mplplots.utils import auto_ticks

from mdpp.analysis.metrics import (
    compute_radius_of_gyration,
    compute_rmsd,
    compute_rmsf,
    compute_sasa,
)
from mdpp.analysis.trajectory import align_trajectory, load_trajectory
from mdpp.plotting.timeseries import plot_rmsd, plot_rmsf, plot_sasa

plt.style.use("mplplots.styles.GraphPadPrism")

In [ ]:
TOPOLOGY_PATH = Path("/path/to/topology.pdb")
TRAJECTORY_PATH = Path("/path/to/trajectory.xtc")
STRIDE = 5

if not TOPOLOGY_PATH.exists() or not TRAJECTORY_PATH.exists():
    raise FileNotFoundError(
        "Update TOPOLOGY_PATH and TRAJECTORY_PATH before running analysis cells."
    )

traj = load_trajectory(
    trajectory_path=TRAJECTORY_PATH,
    topology_path=TOPOLOGY_PATH,
    stride=STRIDE,
    atom_selection="protein",
)
traj = align_trajectory(traj, atom_selection="name CA")

print(f"Frames: {traj.n_frames}, Atoms: {traj.n_atoms}")

## Compute RMSD/RMSF/SASA/Radius of Gyration


In [ ]:
rmsd_result = compute_rmsd(
    traj,
    atom_selection="backbone",
    align=False,
)
rmsf_result = compute_rmsf(
    traj,
    atom_selection="name CA",
    align=False,
)
sasa_result = compute_sasa(
    traj,
    atom_selection="protein",
    mode="residue",
)
rg_result = compute_radius_of_gyration(
    traj,
    atom_selection="protein",
)

## Plot Core Time-Series and Per-Residue Profiles


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), dpi=120)

plot_rmsd(rmsd_result, ax=axes[0, 0], label="trajectory")
axes[0, 0].set_title("RMSD")

plot_rmsf(rmsf_result, ax=axes[0, 1], label="trajectory")
axes[0, 1].set_title("RMSF")

plot_sasa(sasa_result, ax=axes[1, 0], label="sum", aggregate="sum")
axes[1, 0].set_title("SASA (sum over residues)")

axes[1, 1].plot(
    rg_result.time_ns,
    rg_result.radius_gyration_angstrom,
    linewidth=1.5,
    label="trajectory",
)
axes[1, 1].set_xlabel("Time (ns)")
axes[1, 1].set_ylabel("Radius of Gyration (Å)")
axes[1, 1].set_title("Radius of Gyration")
axes[1, 1].legend()

auto_ticks(*axes.ravel())
fig.tight_layout()

## Optional: Mean Per-Residue SASA Profile


In [ ]:
if sasa_result.mode != "residue" or sasa_result.residue_ids is None:
    raise RuntimeError("Set mode='residue' in compute_sasa to plot per-residue profile.")

mean_residue_sasa = np.mean(sasa_result.values_nm2, axis=0)

fig, ax = plt.subplots(figsize=(8, 4), dpi=120)
ax.plot(sasa_result.residue_ids, mean_residue_sasa, linewidth=1.5)
ax.set_xlabel("Residue ID")
ax.set_ylabel("Mean SASA (nm²)")
ax.set_title("Mean Per-Residue SASA")
auto_ticks(ax)
fig.tight_layout()